# 3. Fault 모델 재검증

## 사전 고정 조건

경보 시점은 80·100·120·150시간, 확률 임계값은 0.20, 이동창은 최근 20시간으로 분석 전에 고정한다. 평가자료의 최종 결과와 실제 종료시간은 입력에 사용하지 않는다. 현재 Fault가 10개뿐이므로 새 Fault를 임의 생성해 독립 사례처럼 취급하지 않고, 기존 100개 배치에서 30회 반복 층화 5-fold 교차검증을 실시한다. CSV는 저장하지 않는다.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats


def find_project_root(start=Path.cwd()):
    for root in (start, *start.parents):
        if (root / 'data/interim/merged_data_ko.csv').exists():
            return root
    raise FileNotFoundError('merged_data_ko.csv를 찾을 수 없습니다.')


data = pd.read_csv(find_project_root() / 'data/interim/merged_data_ko.csv')
ALERT_THRESHOLD = 0.20
HORIZONS = [80, 100, 120, 150]
WINDOW_HOURS = 20
print(f'배치 {data["배치번호"].nunique()}개, Fault {data.loc[data["배치번호"] > 90, "배치번호"].nunique()}개')

배치 100개, Fault 10개


### 판단

Fault가 10개라 외부 검증을 대신할 수 없다. 반복 교차검증은 현재 데이터에서의 불안정성을 줄여 보여줄 뿐 신규 공정에 대한 독립 성능을 보장하지 않는다.

In [2]:
level_features = ['산누적량', 'DO평균', 'DO최솟값', 'pH표준편차', 'OUR평균', 'OUR음수비율', 'CO2평균', '전체기질기울기']
dynamic_features = level_features + [
    '최근페니실린기울기', '최근기질기울기', '최근OUR기울기', '최근pH기울기', '최근DO기울기',
    'OUR최근변화량', 'DO최근변화량',
]


def slope(time, values):
    return float(np.polyfit(time, values, 1)[0])


def make_features(horizon):
    rows = []
    for batch_number, batch in data.groupby('배치번호', sort=True):
        batch = batch.sort_values('발효시간(h)')
        observed = batch.loc[batch['발효시간(h)'] <= horizon]
        recent = observed.loc[observed['발효시간(h)'] > horizon - WINDOW_HOURS]
        previous = observed.loc[(observed['발효시간(h)'] > horizon - 2 * WINDOW_HOURS) &
                                (observed['발효시간(h)'] <= horizon - WINDOW_HOURS)]
        time = observed['발효시간(h)'].to_numpy()
        recent_time = recent['발효시간(h)'].to_numpy()
        our = observed['산소소모율(g/min)'].to_numpy()
        rows.append({
            '배치번호': batch_number, 'Fault': int(batch_number > 90),
            '산누적량': np.trapezoid(observed['산투입유량(L/h)'], time),
            'DO평균': observed['용존산소(mg/L)'].mean(), 'DO최솟값': observed['용존산소(mg/L)'].min(),
            'pH표준편차': observed['pH'].std(ddof=1), 'OUR평균': our.mean(), 'OUR음수비율': np.mean(our < 0),
            'CO2평균': observed['배가스이산화탄소(%)'].mean(),
            '전체기질기울기': slope(time, observed['기질농도(g/L)']),
            '최근페니실린기울기': slope(recent_time, recent['페니실린농도(g/L)']),
            '최근기질기울기': slope(recent_time, recent['기질농도(g/L)']),
            '최근OUR기울기': slope(recent_time, recent['산소소모율(g/min)']),
            '최근pH기울기': slope(recent_time, recent['pH']),
            '최근DO기울기': slope(recent_time, recent['용존산소(mg/L)']),
            'OUR최근변화량': recent['산소소모율(g/min)'].mean() - previous['산소소모율(g/min)'].mean(),
            'DO최근변화량': recent['용존산소(mg/L)'].mean() - previous['용존산소(mg/L)'].mean(),
        })
    return pd.DataFrame(rows)


for horizon in HORIZONS:
    feature_data = make_features(horizon)
    print(horizon, '시간:', feature_data.shape, '결측치', int(feature_data[dynamic_features].isna().sum().sum()))

80 시간: (100, 17) 결측치 0
100 시간: (100, 17) 결측치 0
120 시간: (100, 17) 결측치 0
150 시간: (100, 17) 결측치 0


### 판단

모든 시점에서 100개 배치와 15개 동적 특징을 만들었으며 결측치는 없다. 최근 20시간 기울기와 직전 20시간 대비 평균 변화량을 추가해 단일 수준값보다 궤적 변화에 반응하도록 했다.

In [3]:
def fit_ridge_logistic(design, target, penalty=1.0, max_iterations=100):
    coefficients = np.zeros(design.shape[1])
    penalty_matrix = np.eye(design.shape[1])
    penalty_matrix[0, 0] = 0
    for _ in range(max_iterations):
        probability = 1 / (1 + np.exp(-np.clip(design @ coefficients, -30, 30)))
        weight = np.clip(probability * (1 - probability), 1e-8, None)
        gradient = design.T @ (target - probability) - penalty * penalty_matrix @ coefficients
        hessian = design.T @ (weight[:, None] * design) + penalty * penalty_matrix
        step = np.linalg.solve(hessian, gradient)
        coefficients += step
        if np.max(np.abs(step)) < 1e-8:
            break
    return coefficients


def roc_auc(target, score):
    positive = target.sum()
    negative = len(target) - positive
    ranks = stats.rankdata(score)
    return (ranks[target == 1].sum() - positive * (positive + 1) / 2) / (positive * negative)


def average_precision(target, score):
    order = np.argsort(-score)
    ordered_target = target[order]
    precision = np.cumsum(ordered_target) / np.arange(1, len(target) + 1)
    return np.sum(precision * ordered_target) / ordered_target.sum()


def repeated_cross_validated_scores(feature_data, features, repeats=30):
    target = feature_data['Fault'].to_numpy()
    score_sum = np.zeros(len(target))
    score_count = np.zeros(len(target))
    rng = np.random.default_rng(42)
    for _ in range(repeats):
        fold_ids = np.empty(len(target), dtype=int)
        for class_value in [0, 1]:
            indices = np.flatnonzero(target == class_value)
            rng.shuffle(indices)
            fold_ids[indices] = np.arange(len(indices)) % 5
        for fold in range(5):
            train = fold_ids != fold
            test = ~train
            train_values = feature_data.loc[train, features]
            mean = train_values.mean().to_numpy()
            std = train_values.std(ddof=1).replace(0, 1).to_numpy()
            train_design = np.column_stack([np.ones(train.sum()), (train_values.to_numpy() - mean) / std])
            test_design = np.column_stack([np.ones(test.sum()), (feature_data.loc[test, features].to_numpy() - mean) / std])
            coefficients = fit_ridge_logistic(train_design, target[train])
            score = 1 / (1 + np.exp(-np.clip(test_design @ coefficients, -30, 30)))
            score_sum[test] += score
            score_count[test] += 1
    return score_sum / score_count


performance_rows = []
for horizon in HORIZONS:
    feature_data = make_features(horizon)
    target = feature_data['Fault'].to_numpy()
    for model_name, features in [('수준모형', level_features), ('동적모형', dynamic_features)]:
        score = repeated_cross_validated_scores(feature_data, features)
        alert = score >= ALERT_THRESHOLD
        true_positive = np.sum(alert & (target == 1))
        false_positive = np.sum(alert & (target == 0))
        true_negative = np.sum((~alert) & (target == 0))
        false_negative = np.sum((~alert) & (target == 1))
        performance_rows.append({
            '시간': horizon, '모형': model_name, 'ROC_AUC': roc_auc(target, score),
            'PR_AUC': average_precision(target, score),
            '민감도': true_positive / (true_positive + false_negative),
            '특이도': true_negative / (true_negative + false_positive),
            '정밀도': true_positive / (true_positive + false_positive) if true_positive + false_positive else np.nan,
            '오경보수': false_positive,
            '정상1000시간당오경보': false_positive / (90 * horizon) * 1000,
        })
performance = pd.DataFrame(performance_rows)
display(performance.round(6))

,시간,모형,ROC_AUC,PR_AUC,민감도,특이도,정밀도,오경보수,정상1000시간당오경보
0,80,수준모형,0.685556,0.160407,0.2,0.877778,0.153846,11,1.527778
1,80,동적모형,0.718889,0.399282,0.4,0.900000,0.307692,9,1.250000
2,100,수준모형,0.615556,0.224280,0.2,0.911111,0.200000,8,0.888889
3,100,동적모형,0.593333,0.253511,0.3,0.933333,0.333333,6,0.666667
4,120,수준모형,0.860000,0.651026,0.6,0.944444,0.545455,5,0.462963
5,120,동적모형,0.742222,0.635427,0.6,0.933333,0.500000,6,0.555556
6,150,수준모형,0.854444,0.574225,0.6,0.911111,0.428571,8,0.592593
7,150,동적모형,0.890000,0.764914,0.7,0.966667,0.700000,3,0.222222


### 최종 판단과 결론

- 80시간 동적모형은 ROC-AUC 0.719, PR-AUC 0.399로 수준모형보다 좋아졌지만 민감도 0.40, 정밀도 0.31로 조기 경보 성능은 제한적이다.
- 120시간 수준모형은 ROC-AUC 0.860, PR-AUC 0.651, 민감도 0.60으로 150시간보다 이른 유망 시점이다. 동적 특징은 120시간에서는 추가 이득이 없었다.
- 150시간 동적모형이 가장 좋았으며 ROC-AUC 0.890, PR-AUC 0.765, 민감도 0.70, 특이도 0.967, 정밀도 0.70이다. 정상 1,000 배치시간당 오경보는 0.222건이다.
- 최고 모형도 Fault 10개 중 3개를 놓치고 정상 3개를 오경보한다. 현재 10개 Fault를 반복 사용한 내부 검증이므로 운영 배포 근거로는 부족하다.
- 다음 검증은 신규 Fault 또는 별도 시뮬레이션 시나리오를 확보한 뒤 120시간과 150시간, 임계값 0.20을 변경하지 않고 재평가해야 한다. 새 사례를 확보하기 전에는 탐색모형으로만 유지한다.